# CORDEX subset memory failure

This notebook reproduces a CDS WPS workflow that used too much memory while subsetting daily CORDEX `tas` data for 2056–2075.

The request selects every month and every year in the time range, so it effectively keeps the complete 20-year daily time series over the full spatial domain. The result is deliberately **not** opened with `resp.datasets()` in this notebook, because doing so could add client-side memory use to the server-side issue under investigation.


## Original WPS workflow

The payload below is copied from the failing request.


In [ ]:
request = {
    "inputs": {
        "tas": [
            "c3s-cordex.output.EUR-11.CLMcom.MPI-M-MPI-ESM-LR.rcp85."
            "r1i1p1.CLMcom-CCLM4-8-17.v1.day.tas.v20140515"
        ]
    },
    "steps": {
        "subset_tas_1": {
            "run": "subset",
            "in": {
                "collection": "inputs/tas",
                "time_components": (
                    "month:jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec|"
                    "year:2056,2057,2058,2059,2060,2061,2062,2063,2064,"
                    "2065,2066,2067,2068,2069,2070,2071,2072,2073,2074,2075"
                ),
                "time": "2056/2075",
            },
        }
    },
    "outputs": {"output": "subset_tas_1/output"},
    "doc": "workflow",
}

request


## Build the equivalent Rooki workflow

Importing Rooki contacts the configured WPS service. Change `ROOK_URL` here if the reproduction should run against another deployment.


In [ ]:
import json
import os

os.environ["ROOK_URL"] = "http://rook.dkrz.de/wps"

from rooki import operators as ops


In [ ]:
tas = ops.Input("tas", request["inputs"]["tas"])
subset = ops.Subset(
    tas,
    time=request["steps"]["subset_tas_1"]["in"]["time"],
    time_components=request["steps"]["subset_tas_1"]["in"]["time_components"],
)

serialized_request = json.loads(subset._serialise())
assert serialized_request == request
serialized_request


## Reproduce the failure

The next cell submits the full request and may consume substantial memory on the Rook server. Run it only against the deployment being tested.


In [ ]:
resp = subset.orchestrate()
resp.ok, resp.status


## Inspect the response without loading data

If the workflow succeeds, list the output URLs without downloading or opening the NetCDF result. If it fails, displaying `resp` preserves the response details for diagnosis.


In [ ]:
resp


In [ ]:
if resp.ok:
    print("Output URLs (not downloaded):")
    for url in resp.download_urls():
        print(url)
